# NLP Pipeline Introduction Workshop: Team Notebook

**Team:** Sultan Atanda · John Buni · Echezona Oji · Antonio Sainz
**Course:** PROG8245 Machine Learning Programming

| Section | Owner | Rubric focus |
|---|---|---|
| Environment setup, `requirements.txt`, Challenge 1 | Antonio | Environment Setup and Execution |
| Data exploration (after Part 2), Challenge 2 | Eche | Data Loading and Exploration |
| BERT experiments (after Part 6), Challenge 3 | John | Model Execution and Results |
| Challenge 4, README, repository, final collation | Sultan | Code Quality, Documentation, GitHub Submission |

Parts 1 to 6 are the original workshop cells. Team additions are labelled with the owner's name.

### Environment Setup Notes (Antonio)

**How to set up (Windows, VS Code, PowerShell):**

1. `py -m venv .venv`
2. `.\.venv\Scripts\Activate.ps1`
3. `python -m pip install --upgrade pip`
4. `pip install -r requirements.txt`
5. In VS Code, select the `.venv` kernel, then **Restart Kernel** and **Run All**.

**Verified:** Restart Kernel + Run All finished with no errors on Windows (Python 3.12.7) on Nov 9, 2026. Results: vocabulary reduced from 288,723 to 112,264 unique words (61.12%), and BERT answered "What does BERT stand for?" with confidence 0.7050.

**Setup issues found and how they were fixed:**

- **The original `requirements.txt` was saved in UTF-16 and pinned 65 packages, including `pywin32==310`.** `pywin32` only installs on Windows, so the file breaks on macOS and Linux, and most of the 65 lines were indirect dependencies. *Fix:* a new UTF-8 `requirements.txt` that lists only the libraries the notebook imports (nltk, scikit-learn, numpy, scipy, pandas, transformers, torch, plotly, matplotlib, ipykernel, nbformat), covering every team notebook, each pinned to a tested version.
- **`transformers` 5.x removed the `question-answering` pipeline used in Part 6.** Installing the latest version gives `KeyError: 'The task question-answering has been removed from transformers'`. *Fix:* pin `transformers==4.53.0`.
- **First run downloads data.** Part 2 downloads the NLTK data into a local `nltk_data/` folder, Part 2 downloads 20 Newsgroups (about 14 MB), and Part 6 downloads a BERT model (about 250 MB). These folders should not be committed: `.venv/` and `nltk_data/` belong in `.gitignore`.
- **Windows warning when Part 6 downloads the BERT model:** `huggingface_hub` says the machine does not support symlinks, so its cache runs in a "degraded" mode that uses more disk space. It is harmless (the model still loads and answers). *Fix (optional):* turn on Windows Developer Mode, or set the environment variable `HF_HUB_DISABLE_SYMLINKS_WARNING=1` to hide it.
- **`torch` is a large install** (a few GB with its dependencies), so the first `pip install` takes several minutes.

## Workshop - Deconstructing Language for AI Agents

Objective: By the end of this 90-minute workshop, you will understand and have implemented the core components of a Natural Language Processing (NLP) pipeline. You will learn how raw text is systematically processed to become useful, structured data for training an AI agent like Google's BARD.

Duration: 90 minutes

### 1. Introduction: How Does an AI Learn Language?

When you ask an LLM, such as BARD, a question, it seems to just understand you. But that "understanding" is the result of a massive amount of pre-processing.

An AI doesn't read text like a human. It sees text as a sequence of numbers. To an AI, the words "run," "running," and "ran" are initially just different, unrelated data points. The goal of an NLP pipeline is to standardize and simplify text, transforming it into a structured format that a machine can learn patterns from.

Think of it as preparing ingredients for a recipe. You don't just throw a whole carrot into the pot. You wash it, peel it, and chop it. Our "ingredients" are words, and the "recipe" is the AI model we want to train. 

During this workshop, we'll learn the essential pre-processing steps.

### 2. The Raw Material: A Look at Our Text Data

For any NLP task, we need data. We'll be using the "20 Newsgroups" dataset, a classic collection of about 18,000 documents posted to 20 different Usenet newsgroups. It's perfect for us because it's real, messy text with a large vocabulary.

Let's start by loading the data and performing a quick Exploratory Data Analysis (EDA) to understand what we're working with.

Sample EDA: A First Glance at the Data
First, we need to install the necessary libraries. scikit-learn provides the dataset, nltk is our main NLP toolkit, and matplotlib helps us visualize our findings.

In [11]:
#
# Prerequisite installations for the workshop
# To set up the environment, run this script:
# !pip install scikit-learn nltk matplotlib
#
import nltk
import re
import os

# Warning: This download will copy files to your home directory.
# For example, on Linux, it will copy files to ~/.nltk_data.
# In Windows, it will copy files to C:\Users\YourAccount\AppData\Roaming
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')

# A better way to handle the download is to:
# Ensure 'punkt' is available and nltk_data path is set
nltk_data_path = os.path.join(os.getcwd(), 'nltk_data')
print("Downloading 'punkt' tokenizer...")
nltk.download('punkt', download_dir=nltk_data_path, force=True)
print("Downloading 'punkt_tab' tokenizer...")
nltk.download('punkt_tab', download_dir=nltk_data_path, force=True)
print("Downloading 'stopwords' tokenizer...")
nltk.download('stopwords', download_dir=nltk_data_path, force=True)
print("Downloading 'wordnet' tokenizer...")
nltk.download('wordnet', download_dir=nltk_data_path, force=True)

# Always append the custom nltk_data path (if not already present)
if nltk_data_path not in nltk.data.path:
    nltk.data.path.append(nltk_data_path)

# Debugging paths and contents
print("NLTK Data Paths:", nltk.data.path)
print("Contents of nltk_data:", os.listdir(nltk_data_path))


[nltk_data] Downloading package punkt to c:\Users\KING\Desktop\NLP_Pip
[nltk_data]     elineIntroductionWorkshop\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


[nltk_data] Downloading package punkt_tab to c:\Users\KING\Desktop\NLP
[nltk_data]     _PipelineIntroductionWorkshop\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


[nltk_data] Downloading package stopwords to c:\Users\KING\Desktop\NLP
[nltk_data]     _PipelineIntroductionWorkshop\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


[nltk_data] Downloading package wordnet to c:\Users\KING\Desktop\NLP_P
[nltk_data]     ipelineIntroductionWorkshop\nltk_data...


NLTK Data Paths: ['C:\\Users\\KING/nltk_data', 'c:\\Users\\KING\\Desktop\\SingleLayerPerceptrons_Workshop\\.venv\\nltk_data', 'c:\\Users\\KING\\Desktop\\SingleLayerPerceptrons_Workshop\\.venv\\share\\nltk_data', 'c:\\Users\\KING\\Desktop\\SingleLayerPerceptrons_Workshop\\.venv\\lib\\nltk_data', 'C:\\Users\\KING\\AppData\\Roaming\\nltk_data', 'C:\\nltk_data', 'D:\\nltk_data', 'E:\\nltk_data', 'c:\\Users\\KING\\Desktop\\NLP_PipelineIntroductionWorkshop\\nltk_data']
Contents of nltk_data: ['corpora', 'tokenizers']


Now, let's load the data and see what a raw document looks like.

In [12]:
# Load the dataset
from sklearn.datasets import fetch_20newsgroups
newsgroups_data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))

# Let's inspect a single document
print("--- RAW DOCUMENT ---")
sample_doc = newsgroups_data.data[12]
print(sample_doc)

# Let's get some basic stats
all_text = ' '.join(newsgroups_data.data)
words = all_text.split()
unique_words = set(words)

print(f"\n--- INITIAL STATS ---")
print(f"Number of documents: {len(newsgroups_data.data)}")
print(f"Total words (approx.): {len(words)}")
print(f"Total unique words (vocab size): {len(unique_words)}")

--- RAW DOCUMENT ---
930418

Do what thou wilt shall be the whole of the Law. [Honestly.]
The word of Sin is Restriction. [Would I kid you?]


Does one man's words encompass the majestic vision of thousands
of individuals?  Quoting a man is not the same as quoting the
Order.  Taken out of context, words can be interpreted much
differently than had one applied them within the confines of
their original expression.

I think this is the case regarding Hymenaeus Beta, Frater Superior 
of the Order to which I belong.  When he included that bit
from Merlinus X' he did us all a service.  He showed us the extremes
to which Order members have been known to go in their fervor.
I have little knowledge regarding Reuss' background, but surely
he was an unusual man, and he was an important force in the Order 
for many years.

Yet as people change so do Orders change, and while we look back
so carefully at the dirty laundry of O.T.O. remember that this is
only the surface skim and that many perspecti

You'll notice our vocabulary is massive! This is the complexity we need to reduce.

### Data Loading and Exploration (Eche)
For my exploration, I used all 20 Newsgroups documents after the loader removed headers, footers and quoted replies. I counted the documents in each category and measured document length by splitting the remaining text on whitespace. These lengths described the text before our NLP pipeline ran.

I kept empty documents in the analysis so the tables and charts accounted for the whole dataset. My exploration described the data; it did not involve training or evaluating a classifier.

In [13]:
from collections import Counter
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

category_counts = Counter(newsgroups_data.target)
category_table = pd.DataFrame({
    "category": newsgroups_data.target_names,
    "documents": [category_counts[i] for i in range(len(newsgroups_data.target_names))],
})
document_lengths = np.array([len(doc.split()) for doc in newsgroups_data.data])
length_summary = pd.Series(document_lengths, name="words_per_document").describe(
    percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
)
display(category_table)
display(length_summary.to_frame())
print(f"Documents: {len(document_lengths):,}; categories: {len(category_table)}")
print(f"Empty documents: {(document_lengths == 0).sum():,}")
assert category_table.documents.sum() == len(newsgroups_data.data)

category_fig = px.bar(
    category_table.sort_values("documents"), x="documents", y="category",
    orientation="h", title="20 Newsgroups: documents per category",
    labels={"documents": "Number of documents", "category": "Category"},
    height=650,
)
category_fig.update_layout(template="plotly_white")
category_fig.show()

# A log transform keeps the long tail visible without dropping extreme documents.
length_fig = px.histogram(
    x=np.log10(document_lengths + 1), nbins=60,
    title="Document length distribution (all documents)",
    labels={"x": "log10(whitespace word count + 1)", "y": "Documents"},
)
length_fig.update_layout(template="plotly_white", yaxis_title="Number of documents")
length_fig.show()

,category,documents
0,alt.atheism,799
1,comp.graphics,973
2,comp.os.ms-windows.misc,985
3,comp.sys.ibm.pc.hardware,982
4,comp.sys.mac.hardware,963
5,comp.windows.x,988
6,misc.forsale,975
7,rec.autos,990
8,rec.motorcycles,996
9,rec.sport.baseball,994


,words_per_document
count,18846.000000
mean,181.637748
std,501.333800
min,0.000000
25%,40.000000
50%,83.000000
75%,166.000000
90%,331.000000
95%,558.000000
99%,1750.800000


Documents: 18,846; categories: 20
Empty documents: 515


**Observations**
1. **The categories were not exactly balanced.** The category table showed 18,846 documents across 20 groups. I found that `rec.sport.hockey` had the most documents (999), while `talk.religion.misc` had the fewest (628). The largest group had about 1.59 times as many examples as the smallest. Based on this difference, I would check results for each category if I later used this data for classification.
2. **The average length did not describe a typical document very well.** The mean was 181.64 words, but the median was only 83. Half the documents fell between 40 and 166 words, while the longest had 11,765 words. This showed me that a small number of long posts pulled the average upward. I used a log scale for the histogram so I could include those long posts and still see the shorter ones clearly. I measured whitespace word counts, not transformer subword-token counts.
3. **Some documents had no words left after cleaning.** I counted 515 empty documents, or 2.73% of the dataset. I kept them in these results. This reminded me to check what remained after cleaning instead of assuming every record was usable. Before building a classifier, I would inspect these records and decide how to handle those with no text.

### 3. The NLP Pipeline: From Chaos to Structure

We will now build the core components of our pipeline. Each step feeds its output to the next, progressively refining the text.

#### Part 1: Tokenization - Creating the Building Blocks

Concept: Tokenization is the process of breaking down a stream of text into individual units, or tokens. These tokens are the fundamental building blocks for any further analysis. Usually, tokens are words, but they can also be sentences or characters.

Implementation: A Simple Tokenizer
We can start with a basic algorithm that splits text by spaces and removes punctuation.

In [14]:
import re

def simple_tokenizer(text):
  """
  A simple implementation of a word tokenizer.
  1. Splits text by non-word characters.
  2. Returns a list of word tokens.
  """
  # Use regular expression to find all sequences of word characters
  tokens = re.findall(r'\b\w+\b', text)
  return tokens

# Let's tokenize our sample document
raw_tokens = simple_tokenizer(sample_doc)
print("--- TOKENIZATION ---")
print(raw_tokens)

--- TOKENIZATION ---
['930418', 'Do', 'what', 'thou', 'wilt', 'shall', 'be', 'the', 'whole', 'of', 'the', 'Law', 'Honestly', 'The', 'word', 'of', 'Sin', 'is', 'Restriction', 'Would', 'I', 'kid', 'you', 'Does', 'one', 'man', 's', 'words', 'encompass', 'the', 'majestic', 'vision', 'of', 'thousands', 'of', 'individuals', 'Quoting', 'a', 'man', 'is', 'not', 'the', 'same', 'as', 'quoting', 'the', 'Order', 'Taken', 'out', 'of', 'context', 'words', 'can', 'be', 'interpreted', 'much', 'differently', 'than', 'had', 'one', 'applied', 'them', 'within', 'the', 'confines', 'of', 'their', 'original', 'expression', 'I', 'think', 'this', 'is', 'the', 'case', 'regarding', 'Hymenaeus', 'Beta', 'Frater', 'Superior', 'of', 'the', 'Order', 'to', 'which', 'I', 'belong', 'When', 'he', 'included', 'that', 'bit', 'from', 'Merlinus', 'X', 'he', 'did', 'us', 'all', 'a', 'service', 'He', 'showed', 'us', 'the', 'extremes', 'to', 'which', 'Order', 'members', 'have', 'been', 'known', 'to', 'go', 'in', 'their', 'ferv

> Talking Point: "Tokenization is the first and most critical step; it defines the discrete units of meaning—our 'words'—that the machine will learn from."

#### Part 2: Normalization & Stop Words Removal - Cleaning the Noise

Concept: Normalization is about standardizing tokens to treat different forms of a word as the same. The most common step is converting all text to lowercase. Why? Because we don't want the model to think "Apple" (the company) and "apple" (the fruit) are different purely based on capitalization if the context doesn't require it.

Concept: Stop words are extremely common words like "the," "a," "is," "in." They add grammatical structure for humans but often add little semantic value for a machine. Removing them helps the model focus on the words that carry the most meaning.

Implementation: Lowercasing and Filtering

In [15]:
from nltk.corpus import stopwords

def normalize_and_remove_stops(tokens):
    """
    1. Converts all tokens to lowercase (Normalization).
    2. Removes common English stop words.
    """
    # 1. Normalization: Convert to lowercase
    normalized_tokens = [token.lower() for token in tokens]

    # 2. Stop Words Removal
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [token for token in normalized_tokens if token not in stop_words]
    return filtered_tokens

# Process our tokens
cleaned_tokens = normalize_and_remove_stops(raw_tokens)
print("--- NORMALIZATION & STOP WORDS REMOVAL ---")
print(cleaned_tokens)

--- NORMALIZATION & STOP WORDS REMOVAL ---
['930418', 'thou', 'wilt', 'shall', 'whole', 'law', 'honestly', 'word', 'sin', 'restriction', 'would', 'kid', 'one', 'man', 'words', 'encompass', 'majestic', 'vision', 'thousands', 'individuals', 'quoting', 'man', 'quoting', 'order', 'taken', 'context', 'words', 'interpreted', 'much', 'differently', 'one', 'applied', 'within', 'confines', 'original', 'expression', 'think', 'case', 'regarding', 'hymenaeus', 'beta', 'frater', 'superior', 'order', 'belong', 'included', 'bit', 'merlinus', 'x', 'us', 'service', 'showed', 'us', 'extremes', 'order', 'members', 'known', 'go', 'fervor', 'little', 'knowledge', 'regarding', 'reuss', 'background', 'surely', 'unusual', 'man', 'important', 'force', 'order', 'many', 'years', 'yet', 'people', 'change', 'orders', 'change', 'look', 'back', 'carefully', 'dirty', 'laundry', 'remember', 'surface', 'skim', 'many', 'perspectives', 'encompassed', 'extend', 'beyond', 'one', 'individual', 'hope', 'show', 'much', 'room'

> Talking Point (Normalization): "By converting every token to lowercase, we prevent the model from treating the same word differently, drastically reducing the vocabulary size it needs to handle."

> Talking Point (Stop Words): "Removing stop words is like filtering out the static; it lets the model focus on the high-signal words that define the document's topic."

#### Part 3: Stemming - Finding the Root

Concept: Stemming is a process of reducing a word to its root or stem. It does this by applying a crude set of rules to chop off common prefixes or suffixes (affixes). For example, "running," "runs," and "ran" might all be stemmed to the base form "run." This process, known as affix stripping, is heuristic and can sometimes result in non-dictionary words (e.g., "studies" becomes "studi"). However, it's fast and effective for collapsing vocabulary.

Implementation: The Porter Stemmer
We will use the famous Porter Stemming algorithm, available in NLTK.

In [16]:
from nltk.stem import PorterStemmer

def stem_tokens(tokens):
    """
    Applies the Porter Stemming algorithm to a list of tokens.
    This is a form of affix stripping.
    """
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(token) for token in tokens]
    return stemmed_tokens

# Apply stemming to our cleaned tokens
final_tokens = stem_tokens(cleaned_tokens)
print("--- STEMMING (AFFIX STRIPPING) ---")
print(final_tokens)

--- STEMMING (AFFIX STRIPPING) ---
['930418', 'thou', 'wilt', 'shall', 'whole', 'law', 'honestli', 'word', 'sin', 'restrict', 'would', 'kid', 'one', 'man', 'word', 'encompass', 'majest', 'vision', 'thousand', 'individu', 'quot', 'man', 'quot', 'order', 'taken', 'context', 'word', 'interpret', 'much', 'differ', 'one', 'appli', 'within', 'confin', 'origin', 'express', 'think', 'case', 'regard', 'hymenaeu', 'beta', 'frater', 'superior', 'order', 'belong', 'includ', 'bit', 'merlinu', 'x', 'us', 'servic', 'show', 'us', 'extrem', 'order', 'member', 'known', 'go', 'fervor', 'littl', 'knowledg', 'regard', 'reuss', 'background', 'sure', 'unusu', 'man', 'import', 'forc', 'order', 'mani', 'year', 'yet', 'peopl', 'chang', 'order', 'chang', 'look', 'back', 'care', 'dirti', 'laundri', 'rememb', 'surfac', 'skim', 'mani', 'perspect', 'encompass', 'extend', 'beyond', 'one', 'individu', 'hope', 'show', 'much', 'room', 'differ', 'opinion', 'within', 'order', 'perhap', 'test', 'limit', 'let', 'us', 'exami

> Talking Point (Stemming/Affix Stripping): "Stemming aggressively strips word endings to group related terms; it’s a powerful but blunt tool for consolidating our vocabulary down to its core concepts."

#### Part 4: The Final Result & The Big Picture

Let's put it all together and see the transformation.

In [17]:
# The full pipeline
def nlp_pipeline(text):
    tokens = simple_tokenizer(text)
    cleaned_tokens = normalize_and_remove_stops(tokens)
    final_tokens = stem_tokens(cleaned_tokens)
    return final_tokens

# Before
print("--- BEFORE PIPELINE ---")
print(sample_doc)

print("\n" + "="*50 + "\n")

# After
print("--- AFTER PIPELINE ---")
processed_doc = nlp_pipeline(sample_doc)
print(processed_doc)

# Let's re-evaluate our vocabulary size on the entire dataset
all_processed_tokens = nlp_pipeline(' '.join(newsgroups_data.data))
final_vocab_size = len(set(all_processed_tokens))

print(f"\n--- FINAL STATS ---")
print(f"Initial unique words: {len(unique_words)}")
print(f"Final unique words (vocab size): {final_vocab_size}")
print(f"Vocabulary Reduction: {((len(unique_words) - final_vocab_size) / len(unique_words)) * 100:.2f}%")

--- BEFORE PIPELINE ---
930418

Do what thou wilt shall be the whole of the Law. [Honestly.]
The word of Sin is Restriction. [Would I kid you?]


Does one man's words encompass the majestic vision of thousands
of individuals?  Quoting a man is not the same as quoting the
Order.  Taken out of context, words can be interpreted much
differently than had one applied them within the confines of
their original expression.

I think this is the case regarding Hymenaeus Beta, Frater Superior 
of the Order to which I belong.  When he included that bit
from Merlinus X' he did us all a service.  He showed us the extremes
to which Order members have been known to go in their fervor.
I have little knowledge regarding Reuss' background, but surely
he was an unusual man, and he was an important force in the Order 
for many years.

Yet as people change so do Orders change, and while we look back
so carefully at the dirty laundry of O.T.O. remember that this is
only the surface skim and that many perspe

### Part 5: Conclusion 

As you can see, we've taken unstructured, messy human language and converted it into a clean, standardized list of meaningful tokens. We've dramatically reduced the vocabulary size, making it much more efficient for an AI model to process.

This processed data is now ready to be converted into numerical vectors—a process called vectorization (e.g., TF-IDF or Word2Vec)—which is the final step before feeding it into a machine learning model to perform tasks like text classification, sentiment analysis, or even to train a foundational model like BARD.

### Part 6: Fun Activity: Ask BERT a Question!

So far, we've built an NLP pipeline from scratch. This is a fundamental skill, and it shows you the deliberate, step-by-step process of making text useful for machines.

Now, let's jump to the forefront of NLP and use a massive, pre-trained model: BERT (Bidirectional Encoder Representations from Transformers).

What is BERT?
Think of the pipeline we just built. Its goal was to simplify and standardize words. A model like BERT does something far more complex. During its "pre-training" on nearly the entire internet, it learned not just words, but the relationships between words.

Watch this video as an introduction to Transfoemrs to help you understand what BERT is, and where it came from:

[Transformer models and BERT model: Overview](http://www.youtube.com/watch?v=t45S_MwAcOw)


The key is its bidirectional nature. Unlike older models that read text left-to-right, BERT reads the entire sentence at once. This allows it to understand context. For example, it knows that "bank" in "river bank" is different from "bank" in "bank account." The pipeline we built would stem both to "bank" and lose that contextual meaning.

We can use the power of these pre-trained models for specific tasks without needing to train them ourselves. Let's try one of the most popular tasks: Extractive Question Answering.

The Goal: We will give the model a paragraph of text (the context) and a question. The model's job is to find and extract the exact span of text from the context that answers our question.

#### Implementation: The Hugging Face Pipeline 🚀

The transformers library from Hugging Face makes using models like BERT incredibly simple with a tool called pipeline. It handles all the complex tokenization and processing behind the scenes, so we can focus on the results.

In [18]:
#
# Prerequisite: Make sure you have transformers installed!
# Prerequisite: Install the necessary libraries
# The 'transformers' library requires either PyTorch or TensorFlow.
# We'll install torch here.
#
# !pip install transformers torch
#
from transformers import pipeline
import textwrap # Used for nice printing of the context

# 1. Create a Question-Answering pipeline
# This will download a default BERT-based model fine-tuned for this task.
# No authentication is needed!
qa_pipeline = pipeline("question-answering")

# 2. Define the context
# This is the body of text the model will read to find the answer.
context = """
BERT, which stands for Bidirectional Encoder Representations from Transformers,
is a large language model developed by Google. It was a major breakthrough in
the field of Natural Language Processing because it was the first model to look
at the full context of a word by examining the words that come before and after it.
This bidirectional capability allows it to grasp complex nuances in language.
The model was pre-trained on a massive amount of text data from sources like
Wikipedia and the BooksCorpus.
"""

# 3. Ask a question!
# The answer to this question must be present in the context above.
question = "What does BERT stand for?"

# 4. Get the answer
result = qa_pipeline(question=question, context=context)

# Let's print the results nicely
print("--- BERT Question-Answering ---")
print(f"❓ Question: {question}")
print(f"✅ Answer: '{result['answer']}'")
print(f"Confidence Score: {result['score']:.4f}")

# --- Now, it's your turn! ---
#
# Try asking different questions. For example:
# - "Who developed BERT?"
# - "What capability allows BERT to grasp nuances?"
# - "What was the model pre-trained on?"
#
# You can even change the context entirely to a paragraph about a
# topic you find interesting and ask questions about it!

print("\n--- Try it yourself! ---")
# Example of another question
my_question = "Who developed BERT?"
my_result = qa_pipeline(question=my_question, context=context)

print(f"❓ My Question: {my_question}")
print(f"✅ Answer: '{my_result['answer']}' with score {my_result['score']:.4f}")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


--- BERT Question-Answering ---
❓ Question: What does BERT stand for?
✅ Answer: 'Bidirectional Encoder Representations from Transformers'
Confidence Score: 0.7050

--- Try it yourself! ---
❓ My Question: Who developed BERT?
✅ Answer: 'Google' with score 0.9778


⚠️ A Quick Word on Model Limitations
It's important to understand that the BERT model we're using is not a general-purpose chatbot. It has been fine-tuned for a very specific task: extractive question answering.

This means its only job is to find and extract the exact span of text from the context that best answers the question. It does not understand abstract concepts, perform reasoning, or generate new sentences.

Watch what happens when we ask it about a joke.

In [19]:
# Example of another question
my_question = "Can BERT tell jokes?"
my_result = qa_pipeline(question=my_question, context=context)

print(f"❓ My Question: {my_question}")
print(f"✅ Answer: '{my_result['answer']}' with score {my_result['score']:.4f}")

❓ My Question: Can BERT tell jokes?
✅ Answer: 'Bidirectional Encoder Representations from Transformers' with score 0.0830


In [20]:
# The context contains a simple joke
joke_context = "I told my computer I needed a break, and now it won’t stop sending me Kit-Kat ads."

# The question requires understanding humor, not just extracting text
joke_question = "Why is this joke funny?"

# Let's see how the extractive model handles it
joke_result = qa_pipeline(question=joke_question, context=joke_context)

print(f"❓ Question: {joke_question}")
print(f"✅ Answer: '{joke_result['answer']}'")
print(f"Confidence Score: {joke_result['score']:.4f}")

❓ Question: Why is this joke funny?
✅ Answer: 'I needed a break'
Confidence Score: 0.0999


As you can see, the model fails. It extracts a piece of the sentence but completely misses the pun and the humorous concept. The low confidence score also shows it is struggling.

This model will generally fail at questions that require:

Reasoning or Synthesis: It can't combine multiple pieces of information.

"Yes/No" Answers: Unless the words "yes" or "no" are explicitly in the text.

Abstract Concepts: Like humor, sarcasm, or summarizing the main idea.

This is a key difference between an extractive model like this one and a generative AI like BARD, which is designed to understand these nuances and generate new, explanatory text.

**A note on the Part 6 model (Eche):** The selected checkpoint was DistilBERT fine-tuned for extractive question answering. Its score did not guarantee a correct answer, especially when the context did not contain an answer. BERT was pretrained on BooksCorpus and English Wikipedia, rather than the entire internet.

### BERT Experiments: Model Execution and Results (John)

In this experiment, I test the existing BERT question-answering pipeline using four additional questions.

Three questions ask for information explicitly stated in the original context. The fourth asks for information that is missing from the context.

The goal is to compare how well an extractive question-answering model handles answerable and unanswerable questions and identify its limitations.

In [21]:

# John's BERT question-answering experiments
# Reuse the qa_pipeline and context from Part 6

test_questions = [
    {
        "question": "Which company developed BERT?",
        "expected": "Google"
    },
    {
        "question": "What two sources were used to pre-train BERT?",
        "expected": "Wikipedia and the BooksCorpus"
    },
    {
        "question": "What does BERT examine to understand a word?",
        "expected": "The words that come before and after it"
    },
    {
        "question": "In what year was BERT released?",
        "expected": "Not stated in the context"
    }
]

print("--- John's BERT Experiments ---")

for number, test in enumerate(test_questions, start=1):
    result = qa_pipeline(
        question=test["question"],
        context=context
    )

    print(f"\nExperiment {number}")
    print(f"Question: {test['question']}")
    print(f"Expected: {test['expected']}")
    print(f"Model answer: {result['answer']}")
    print(f"Confidence score: {result['score']:.4f}")
    print("-" * 50)


--- John's BERT Experiments ---

Experiment 1
Question: Which company developed BERT?
Expected: Google
Model answer: Google
Confidence score: 0.9886
--------------------------------------------------

Experiment 2
Question: What two sources were used to pre-train BERT?
Expected: Wikipedia and the BooksCorpus
Model answer: Wikipedia and the BooksCorpus
Confidence score: 0.9493
--------------------------------------------------

Experiment 3
Question: What does BERT examine to understand a word?
Expected: The words that come before and after it
Model answer: the words that come before and after it
Confidence score: 0.4979
--------------------------------------------------

Experiment 4
Question: In what year was BERT released?
Expected: Not stated in the context
Model answer: Bidirectional Encoder Representations from Transformers
Confidence score: 0.2326
--------------------------------------------------


### BERT Results and Interpretation (John)

**Experiment 1 — Model Developer**

I asked which company developed BERT. The model correctly answered **Google**, with a confidence score of **0.9886 (98.86%)**.

This was a straightforward question because the answer was explicitly stated in the context.

**Experiment 2 — Pre-training Sources**

I asked which two sources were used to pre-train BERT. The model correctly answered **Wikipedia and the BooksCorpus**, with a confidence score of **0.9493 (94.93%)**.

This demonstrates that the model can identify and extract multiple pieces of related information from a passage.

**Experiment 3 — Bidirectional Context**

I asked what BERT examines to understand a word. The model correctly answered **"the words that come before and after it"**, with a confidence score of **0.4979 (49.79%)**.

Although the answer was correct, the confidence score was noticeably lower than in the first two experiments. This shows that a lower score does not automatically mean the answer is incorrect.

**Experiment 4 — Missing Information (Model Failure)**

I asked what year BERT was released. The correct response, based on the supplied passage, would be that the information was not provided.

Instead, the model returned **"Bidirectional Encoder Representations from Transformers"**, with a confidence score of **0.2326 (23.26%)**.

This response did not answer the question. It illustrates a limitation of the extractive question-answering pipeline: when information is missing, the model may still select an unrelated phrase from the context rather than explicitly stating that the answer is unavailable.

**Overall Interpretation**

The BERT question-answering model answered **3 out of 4 questions correctly (75%)** in my small experiment.

It performed well when the answers were explicitly available in the passage. However, it struggled when asked a question that could not be answered from the supplied context.

Unlike Bag-of-Words, which represents documents using word counts, BERT uses contextual information to interpret relationships between words.

These experiments show that contextual language models can provide more meaningful text understanding than traditional word-count approaches, but their predictions still require evaluation, particularly when the requested information is absent.

The confidence scores also should not be treated as guaranteed probabilities of correctness.

### Part 7: Challenges

You have now seen a complete introductory NLP pipeline in action. But real NLP systems are rarely “finished.”  
There is almost always a missing piece, a trade-off, or a design decision that could be improved.

In this section, you will tackle **four beginner-friendly challenges** that push the current notebook a little further.  
Each challenge asks you to investigate a limitation, improve part of the pipeline, or test whether a different idea works better.

These are not just coding exercises — they are the kind of questions people ask when building real machine learning systems:

- Are we removing useful information by accident?
- Are we simplifying words too aggressively?
- How can we turn processed text into features a model could actually use?
- How does tokenization affect cost when we use commercial LLMs?

Take your time, experiment, and discuss what you notice.

#### Challenge 1: Are We Throwing Away Important Words? (Antonio)

Our current pipeline removes common stop words such as **"the"**, **"is"**, and **"and"**.  
That usually helps reduce noise, but it can also create a problem.

Some so-called “small” words actually carry **big meaning**.  
For example, the sentence:

> “This movie is **not** good.”

changes completely if we remove the word **not**.

Your challenge is to improve the stop-word removal step so that it keeps a few important **negation words** such as:

- `not`
- `no`
- `never`

Then test the result on a short sentence and see how the meaning changes when these words are preserved.

**Goal:** make the pipeline a little smarter by keeping words that are small, but important.

In [22]:
# Challenge 1 Scaffold:
# Modify stop-word removal so that important negation words are preserved.

test_sentence = "This movie is not good, and I would never recommend it to anyone."

def normalize_and_remove_stops_keep_negation(tokens):
    """
    1. Converts tokens to lowercase.
    2. Removes many common stop words.
    3. Keeps important negation words such as 'not', 'no', and 'never'.
    """
    # Convert each token to lowercase
    normalized_tokens = [token.lower() for token in tokens]

    stop_words = set(stopwords.words('english'))

    # Negation words we want to keep because they flip the meaning of a sentence
    negation_words = {"not", "no", "never"}

    # Update stop_words so the negation words are NOT removed
    # (set difference: every stop word except the negation words)
    stop_words = stop_words - negation_words

    # Filter the tokens using the updated stop_words set
    filtered_tokens = [token for token in normalized_tokens if token not in stop_words]

    return filtered_tokens

# Try the original pipeline logic on the test sentence
original_tokens = simple_tokenizer(test_sentence)
original_cleaned = normalize_and_remove_stops(original_tokens)

# Try your improved version
improved_cleaned = normalize_and_remove_stops_keep_negation(original_tokens)

print("--- Original cleaned tokens ---")
print(original_cleaned)

print("\n--- Improved cleaned tokens (keep negation) ---")
print(improved_cleaned)

# Compare the two outputs: which words did the improved version preserve?
preserved_words = [token for token in improved_cleaned if token not in original_cleaned]
print("\n--- Words preserved only by the improved version ---")
print(preserved_words)

# Is each negation word in NLTK's default English stop-word list?
default_stop_words = set(stopwords.words('english'))
print("\n--- Is each negation word in NLTK's default stop-word list? ---")
for word in sorted({"not", "no", "never"}):
    print(f"{word}: {word in default_stop_words}")


--- Original cleaned tokens ---
['movie', 'good', 'would', 'never', 'recommend', 'anyone']

--- Improved cleaned tokens (keep negation) ---
['movie', 'not', 'good', 'would', 'never', 'recommend', 'anyone']

--- Words preserved only by the improved version ---
['not']

--- Is each negation word in NLTK's default stop-word list? ---
never: False
no: True
not: True


**Talking Points — Challenge 1**

Write your observations below:

- What words disappeared in the original version?
- Which negation words did your improved version preserve?
- How could removing words like `not` or `never` confuse a sentiment analysis model?
- Which line(s) of code were most important to solving this challenge?

> Your notes here:
>
> - **Words that disappeared in the original version:** `this`, `is`, `not`, `and`, `i`, `it`, `to`. Most of them are harmless grammar words, but `not` is the one that carries the meaning of the sentence. The original output reads `movie good would never recommend anyone`, which looks like a *positive* review of the movie.
> - **Negation words preserved by the improved version:** `not` is now kept, so the output reads `movie not good would never recommend anyone`. A surprise: `never` was already kept by the original version, because it is **not** in NLTK's default English stop-word list, while `not` and `no` are. This shows that stop-word lists are not neutral: we should check what a list contains before trusting it.
> - **Why it matters for sentiment analysis:** a model that only sees `good` will learn a positive signal from a negative review. Removing `not` or `no` can flip the label of a sentence, which adds noise to training data and makes the model wrong on exactly the cases that matter (complaints, warnings, refusals).
> - **Most important line of code:** `stop_words = stop_words - negation_words`. This set difference builds a custom stop-word list that keeps the negation words, and the filter `[token for token in normalized_tokens if token not in stop_words]` then uses it. The lesson: the stop-word list should be adapted to the task instead of used as-is.


#### Challenge 2: Is Stemming Too Aggressive? (Eche)

Our notebook currently uses **stemming** to reduce words to a simpler form.  
That is useful, but stemming can sometimes be a little rough.

For example:

- `studies` might become `studi`
- `running` might become `run`
- `better` may not behave the way you expect

This raises a good beginner question:

> Is stemming always the best choice?

A more language-aware alternative is **lemmatization**, which tries to convert words into a proper dictionary base form.

Your challenge is to compare **stemming** and **lemmatization** on a small list of words and reflect on which output seems easier for humans to interpret.

**Goal:** explore the trade-off between a fast, simple method and a more linguistically meaningful method.

I used the same six example words for both methods so I could compare their outputs directly. First, I followed the scaffold and called `lemmatize(word)` without a part-of-speech label. That treated each word as a **noun**.

I also included a second lemma column with manually chosen parts of speech. This let me check what happened when `running` and `playing` were treated as verbs and `better` as an adjective. For words in a full sentence, I would need the context to decide which label to use.

In [23]:
from nltk.stem import WordNetLemmatizer

example_words = ["running", "studies", "better", "cars", "wolves", "playing"]
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed_words = []
lemmatized_words = []
for word in example_words:
    stemmed_words.append(stemmer.stem(word))
for word in example_words:
    lemmatized_words.append(lemmatizer.lemmatize(word))

print("Original words:", example_words)
print("Porter stems:", stemmed_words)
print("Default noun lemmas:", lemmatized_words)

chosen_pos = ["v", "n", "a", "n", "n", "v"]
pos_lemmas = [lemmatizer.lemmatize(word, pos=pos)
              for word, pos in zip(example_words, chosen_pos)]
comparison = pd.DataFrame({
    "original": example_words,
    "Porter stem": stemmed_words,
    "default noun lemma": lemmatized_words,
    "chosen POS": chosen_pos,
    "POS-aware lemma": pos_lemmas,
})
display(comparison)
assert len(stemmed_words) == len(lemmatized_words) == len(example_words)

Original words: ['running', 'studies', 'better', 'cars', 'wolves', 'playing']
Porter stems: ['run', 'studi', 'better', 'car', 'wolv', 'play']
Default noun lemmas: ['running', 'study', 'better', 'car', 'wolf', 'playing']


,original,Porter stem,default noun lemma,chosen POS,POS-aware lemma
0,running,run,running,v,run
1,studies,studi,study,n,study
2,better,better,better,a,good
3,cars,car,car,n,car
4,wolves,wolv,wolf,n,wolf
5,playing,play,playing,v,play


**Talking Points: Challenge 2**
- **Which words looked strange after stemming?** The clearest examples for me were `studies → studi` and `wolves → wolv`. I could still guess the original words, but the stems did not look like words I would normally write in a sentence. The other stems in my output were `run`, `better`, `car`, and `play`.
- **Which words looked more natural after lemmatization?** I found `study` and `wolf` easier to read than `studi` and `wolv`. Both methods turned `cars` into `car`. One detail that stood out was that default lemmatization left `running`, `playing`, and `better` unchanged. When I supplied the intended parts of speech, the results became `run`, `play`, and `good`. Switching to a lemmatizer alone did not give all the base forms I was looking for.
- **Why might I still choose stemming?** The comparison showed why stemming could be a simple starting point for search or document classification: it applied rules without requiring a dictionary or part-of-speech labels. Readable feature names would not necessarily be my main priority for those tasks. I did not measure speed or model accuracy in this comparison, so I could not conclude that stemming performed better based on these six words.
- **What did this teach me about trade-offs in NLP?** I learned that readable output and useful model features needed separate consideration. I preferred the lemma table for explaining the results because it produced familiar words such as `study` and `wolf`. The extra POS-aware column showed me why grammatical information mattered for lemmatization. For a future machine learning task, I would compare both methods on that task before choosing.

#### Challenge 3: How Do We Turn Cleaned Text into Features for Machine Learning? (John)

So far, we have cleaned text and produced processed tokens.  
That is an important step, but machine learning models still need **numbers**, not just words.

One common beginner-friendly solution is a **Bag-of-Words** representation.  
The idea is simple:

- count how often words appear
- choose the most common terms
- represent each document using those counts

This is one of the classic bridges between **NLP** and **Machine Learning**.

Your challenge is to build a small Bag-of-Words feature view from the processed dataset and inspect the most frequent tokens.

**Goal:** connect text preprocessing to the kind of numerical features a model could actually learn from.

In [24]:
# Challenge 3
# Bag-of-Words: frequent tokens and a document feature vector

from collections import Counter

# Use the first 50 documents from the 20 Newsgroups dataset
small_corpus = newsgroups_data.data[:50]

# Apply the existing NLP pipeline to each document
processed_documents = []

for doc in small_corpus:
    processed_tokens = nlp_pipeline(doc)
    processed_documents.append(processed_tokens)

# Combine the processed tokens from all 50 documents
all_tokens = []

for token_list in processed_documents:
    all_tokens.extend(token_list)

# Count token frequencies across the small corpus
token_counts = Counter(all_tokens)

# Display the 15 most frequent tokens
top_15 = token_counts.most_common(15)

print("--- Top 15 Tokens in the Small Corpus ---")
print(f"{'Token':<20}Frequency")
print("-" * 30)

for token, count in top_15:
    print(f"{token:<20}{count}")

# Optional extension: Bag-of-Words vector for one document

# Select the first processed document
selected_doc = processed_documents[0]

# Use the five most frequent corpus tokens as our vocabulary
vocabulary = [token for token, count in top_15[:5]]

# Count each vocabulary token in the selected document
doc_counts = Counter(selected_doc)
bow_vector = {}

for token in vocabulary:
    bow_vector[token] = doc_counts[token]

print("\n--- Example Bag-of-Words Vector ---")
print("Document index: 0")
print("Vocabulary:", vocabulary)
print("BoW vector:", bow_vector)


# Find a document containing at least one vocabulary token
for doc_index, tokens in enumerate(processed_documents):
    counts = Counter(tokens)
    vector = [counts[token] for token in vocabulary]

    if any(vector):
        print("\n--- Additional Nonzero Bag-of-Words Example ---")
        print("Document index:", doc_index)
        print("Vocabulary:", vocabulary)
        print("BoW vector:", dict(zip(vocabulary, vector)))
        break

--- Top 15 Tokens in the Small Corpus ---
Token               Frequency
------------------------------
det                 45
nyr                 40
tor                 39
bo                  37
mtl                 33
la                  30
chi                 30
order               26
one                 25
pit                 24
time                23
use                 22
edm                 22
phi                 21
1                   20

--- Example Bag-of-Words Vector ---
Document index: 0
Vocabulary: ['det', 'nyr', 'tor', 'bo', 'mtl']
BoW vector: {'det': 0, 'nyr': 0, 'tor': 0, 'bo': 0, 'mtl': 0}

--- Additional Nonzero Bag-of-Words Example ---
Document index: 7
Vocabulary: ['det', 'nyr', 'tor', 'bo', 'mtl']
BoW vector: {'det': 0, 'nyr': 0, 'tor': 0, 'bo': 2, 'mtl': 0}


**Talking Points: Challenge 3**

**1. Which processed words appeared most often?**

The five most frequent tokens were `det` (45), `nyr` (40), `tor` (39), `bo` (37), and `mtl` (33). Other frequent tokens included `la` (30), `chi` (30), `order` (26), `one` (25), and `pit` (24).

**2. Were the most common words meaningful, surprising, or too generic?**

Many of the most frequent tokens appear to be abbreviations associated with NHL hockey teams, such as `det` (Detroit), `nyr` (New York Rangers), and `tor` (Toronto). This suggests that hockey-related discussions are prominent in the sampled documents.

However, words such as `one`, `time`, and `use` are relatively generic and may be less useful for identifying a document's topic. The results show that high-frequency tokens can reflect the subject matter of a dataset, but they are not necessarily the most informative features for classification.

**3. How does Bag-of-Words help a machine learning model?**

Bag-of-Words converts text into numerical feature vectors by counting how often selected vocabulary terms appear in each document.

For example, my five-token vocabulary was `['det', 'nyr', 'tor', 'bo', 'mtl']`. The first document produced the vector `[0, 0, 0, 0, 0]`, meaning none of these five terms appeared in that document after preprocessing.

A machine learning model can use these vectors as numerical inputs for classification or other NLP tasks.

**4. What information is lost when we reduce a document to word counts?**

Bag-of-Words ignores word order, grammar, and context. For example, "dog bites man" and "man bites dog" produce the same word counts despite having different meanings.

It also does not recognize that different words may express similar meanings. Additionally, restricting the vocabulary to only five tokens can discard most of the information in a document, as demonstrated by my all-zero vector.

**Observation**

Although the code successfully generated the top 15 tokens and a Bag-of-Words vector, the first document contained none of the five selected vocabulary terms. This illustrates how a limited vocabulary can produce sparse feature vectors and fail to represent the content of certain documents.

#### Challenge 4: Why Does Tokenization Matter for LLM Billing? (Sultan)

When we use a commercial Large Language Model, we are usually **not billed per prompt** or per paragraph.  
We are billed by **tokens**.

A token is a chunk of text. Sometimes it is a whole word, sometimes part of a word, and sometimes punctuation.  
That means two prompts that look very similar to a human can still produce **different token counts, costs, and response times**.

This makes tokenization more than just a preprocessing step.  
It becomes part of a real engineering decision:

- How much text should we send?
- Should we remove stop words?
- Should we simplify words first?
- Does a Bag-of-Words view reduce meaning too much?
- How much extra cost comes from the **prompt** compared with the **model's answer**?

For this reflection challenge, imagine you are preparing a long educational prompt for a commercial LLM.  
Use the prompt below as your sample text, then explore how the token count changes if you:

1. remove stop words  
2. apply stemming or lemmatization  
3. convert the text into a Bag-of-Words style view  

You may also compare your observations with external tools such as:

- `https://pricepertoken.com/`
- `https://pricepertoken.com/token-counter`

A real example shared for this workshop showed that two models could receive almost the same prompt, yet differ a lot in **completion length, latency, and cost**.  
That is a useful reminder: **tokenization affects billing, but the model's response style matters too.**

**Goal:** reflect on how preprocessing choices may reduce token counts, and also think critically about what is lost when we compress a prompt too aggressively.

In [25]:
# Challenge 4: how preprocessing changes prompt size and LLM cost

from collections import Counter
import re
import pandas as pd
import plotly.express as px
from IPython.display import display

workshop_prompt = """
I am a professor of Machine Learning at the college level. You are an experienced
Machine Learning Engineer and Data Scientist. I am developing a workshop on time series
using Jupyter Notebooks as a project to help train my students in an active learning environment.

In this case, I want them to learn Data and Time Series Analysis, the Moving Averages calculation,
Decomposition Models, how to Deseasonalize a Time Series, Autoregressive Models, ARMA Models,
ARIMA Models, the Augmented Dickey-Fuller test, and Machine Learning application design patterns,
all simultaneously.

The workshop is based on active learning and requires them to read, brainstorm, write code,
reflect on model performance, and compare results with their peers.
""".strip()


def simple_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


def remove_stop_words(tokens):
    stop_words = set(stopwords.words("english"))
    return [token for token in tokens if token not in stop_words]


def apply_stemming(tokens):
    stemmer = PorterStemmer()
    return [stemmer.stem(token) for token in tokens]


def bag_of_words_view(tokens):
    return Counter(tokens)


baseline_tokens = simple_tokenize(workshop_prompt)
print("Baseline token count:", len(baseline_tokens))
print("First 25 baseline tokens:", baseline_tokens[:25])

tokens_no_stops = remove_stop_words(baseline_tokens)
print("\nToken count after stop-word removal:", len(tokens_no_stops))

stemmed_tokens = apply_stemming(tokens_no_stops)
print("Token count after stemming:", len(stemmed_tokens))

bow_counts = bag_of_words_view(stemmed_tokens)
top_15 = bow_counts.most_common(15)
print("\nTop 15 Bag-of-Words terms:")
print(top_15)

price_per_1k_tokens = 0.008
baseline_cost = len(baseline_tokens) / 1000 * price_per_1k_tokens
processed_cost = len(stemmed_tokens) / 1000 * price_per_1k_tokens
print(f"\nEstimated baseline prompt cost: ${baseline_cost:.6f}")
print(f"Estimated processed prompt cost: ${processed_cost:.6f}")

stages = pd.DataFrame({
    "stage": ["Baseline", "Stop words removed", "Stemmed", "Bag-of-Words (unique terms)"],
    "tokens": [len(baseline_tokens), len(tokens_no_stops), len(stemmed_tokens), len(bow_counts)],
})
stages["reduction_vs_baseline_%"] = (1 - stages["tokens"] / len(baseline_tokens)) * 100
stages["prompt_cost_usd"] = stages["tokens"] / 1000 * price_per_1k_tokens
display(stages.round({"reduction_vs_baseline_%": 1, "prompt_cost_usd": 6}))

stage_fig = px.bar(
    stages, x="stage", y="tokens", text="tokens", color="stage",
    title="Prompt size after each preprocessing step (word-level tokens)",
    labels={"stage": "", "tokens": "Tokens"},
)
stage_fig.update_layout(template="plotly_white", showlegend=False)
stage_fig.show()

# Output tokens are usually priced higher than input tokens; 4x is an assumed ratio for illustration.
output_price_per_1k_tokens = price_per_1k_tokens * 4
billing = pd.DataFrame({"completion_tokens": [200, 800, 2000]})
billing["prompt_cost_usd"] = baseline_cost
billing["completion_cost_usd"] = billing["completion_tokens"] / 1000 * output_price_per_1k_tokens
billing["total_cost_usd"] = billing["prompt_cost_usd"] + billing["completion_cost_usd"]
billing["prompt_share_%"] = billing["prompt_cost_usd"] / billing["total_cost_usd"] * 100
billing["bill_saved_by_preprocessing_%"] = (baseline_cost - processed_cost) / billing["total_cost_usd"] * 100
display(billing.round(6).round({"prompt_share_%": 2, "bill_saved_by_preprocessing_%": 2}))

Baseline token count: 114
First 25 baseline tokens: ['i', 'am', 'a', 'professor', 'of', 'machine', 'learning', 'at', 'the', 'college', 'level', 'you', 'are', 'an', 'experienced', 'machine', 'learning', 'engineer', 'and', 'data', 'scientist', 'i', 'am', 'developing', 'a']

Token count after stop-word removal: 71
Token count after stemming: 71

Top 15 Bag-of-Words terms:
[('learn', 6), ('model', 5), ('machin', 3), ('time', 3), ('seri', 3), ('data', 2), ('workshop', 2), ('activ', 2), ('professor', 1), ('colleg', 1), ('level', 1), ('experienc', 1), ('engin', 1), ('scientist', 1), ('develop', 1)]

Estimated baseline prompt cost: $0.000912
Estimated processed prompt cost: $0.000568


,stage,tokens,reduction_vs_baseline_%,prompt_cost_usd
0,Baseline,114,0.0,0.000912
1,Stop words removed,71,37.7,0.000568
2,Stemmed,71,37.7,0.000568
3,Bag-of-Words (unique terms),53,53.5,0.000424


,completion_tokens,prompt_cost_usd,completion_cost_usd,total_cost_usd,prompt_share_%,bill_saved_by_preprocessing_%
0,200,0.000912,0.0064,0.007312,12.47,4.70
1,800,0.000912,0.0256,0.026512,3.44,1.30
2,2000,0.000912,0.0640,0.064912,1.40,0.53


**Talking Points: Challenge 4**

- **Baseline size.** With the simple word tokenizer the prompt has **114 tokens**, made from 740 characters. A commercial tokenizer would report more than this. It splits text into subword pieces, so uncommon terms such as `deseasonalize`, `autoregressive` and `dickey` turn into several tokens each, and punctuation is counted as well. Our count is a lower bound and is useful for comparing steps against each other. It is not the number that would appear on an invoice.
- **Stop-word removal.** The prompt drops from 114 to **71 tokens (43 fewer, about 37.7%)**. Every saved token was a function word such as `i`, `am`, `a`, `the`, `to`, `and`, `them` or `how`. This was the biggest saving of any step.
- **Stemming.** The count stays at **71**. Stemming rewrites each token (`learning → learn`, `models → model`, `series → seri`, `machine → machin`) but never removes one, so it changes the shape of the words, not how many there are. Lemmatization also maps one word to one word, so it would not reduce the count either.
- **Bag-of-Words.** About 53 unique terms remain. The most frequent are `learn` (6), `model` (5), and `machin`, `time` and `seri` (3 each). The counts show what the prompt is about: machine learning models for time series. They cannot show who is speaking, what is being asked or in what order things should happen.
- **Would we send the Bag-of-Words version to an LLM?** No. *"I am a professor… You are an experienced Machine Learning Engineer"* carries the role, the audience and the task. Once it is turned into `professor, machin, learn, engin, …` the model can no longer tell who is asking for what, so it has to guess. That costs more in poor answers and retries than the few tokens it saves. The same reasoning applies to stemmed text, because `seri` and `deseasonaliz` are harder for the model to read than the original words.
- **Prompt or completion tokens: which matters more for billing?** For this prompt, the completion does. Assuming output is priced at 4× the input rate, an 800-token answer costs about $0.0256, while the whole 114-token prompt costs $0.000912, so the prompt is only about **3.4%** of the bill. Removing the stop words saves about **1.3%** of the total. With a 2,000-token answer the saving falls to about 0.5%. This matches the example shared for the workshop: models given almost the same prompt ended up with very different bills because their answers differed in length.
- **Which step saved the most without damaging the meaning?** Stop-word removal saved the most tokens, but it also dropped words the instructions depend on (`how`, `them`, `all`), and as Antonio showed in Challenge 1, it can drop `not` and `no` as well. In practice the safer savings come from editing the prompt itself: removing repeated phrases, sending only the context that is needed, and asking for a shorter answer, since the answer is where most of the cost is.

### Team Summary

- **Pipeline effect:** tokenization, lowercasing, stop-word removal and stemming reduce the 20 Newsgroups vocabulary from 288,723 to 112,264 unique words (61.12%).
- **Challenge 1:** a generic stop-word list removes `not`, which turns *"not good"* into *"good"*. Stop-word lists should be adapted to the task.
- **Challenge 2:** stemming is simple and needs no dictionary, but it produces non-words (`studi`, `wolv`). Lemmatization gives readable words, but only when it is given the part of speech.
- **Challenge 3:** Bag-of-Words turns documents into count vectors that a model can use, but it loses word order and context. With a small vocabulary, many documents end up as all-zero vectors.
- **Challenge 4:** preprocessing cut the prompt by about 38%, but the completion dominates the bill. Clear, concise prompts save more money than compressed text that loses meaning.
- **BERT (Part 6):** the extractive model answers well when the answer is written in the context. When it is not, it still returns a span from the text, so its score is not a guarantee that the answer is correct.